**TRABAJO PRÁCTICO 3**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import numpy as np
import re
from pandas.api.types import is_categorical_dtype
from sklearn.model_selection import train_test_split
from scipy import stats
import statsmodels.api as sm
from numpy.linalg import LinAlgError
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, accuracy_score, recall_score
)

In [ ]:
#cargamos la base de datos respondieron
respondieron = pd.read_excel("respondieron.xlsx")
respondieron

In [ ]:
columnas_seleccionadas = ["ANO4","AGLOMERADO","PP3E_TOT","CH04","CH06","CH07","CH08","NIVEL_ED","ESTADO","CAT_INAC","CH09","CH03","pobreza"]

prueba = respondieron[columnas_seleccionadas]

In [ ]:
prueba

In [ ]:
df = prueba.copy()

# 1. SEXO → mujer

if "CH04" in df.columns:
    df["mujer"] = (df["CH04"] == 2).astype(int)
    df.drop(columns=["CH04"], inplace=True)

# 2. ESTADO CIVIL (CH07) → en_pareja
# 1=Unido, 2=Casado → en pareja
# 3=Separado, 4=Viudo, 5=Soltero → no en pareja

if "CH07" in df.columns:
    df["en_pareja"] = df["CH07"].isin([1, 2]).astype(int)
    df.drop(columns=["CH07"], inplace=True)

# 3. COBERTURA MÉDICA (CH08) → tiene_cobertura
# (1,2,3,12,13,23,123) = tiene / (4,9) = no tiene

def tiene_cob(x):
    try:
        s = str(int(x))
        return any(d in s for d in ["1","2","3"])
    except:
        return False

if "CH08" in df.columns:
    df["tiene_cobertura"] = df["CH08"].apply(tiene_cob).astype(int)
    df.drop(columns=["CH08"], inplace=True)

# 4. NIVEL EDUCATIVO (NIVEL_ED) → edu_secundario / edu_superior
# 1–3 = primario o menos / 4–5 = secundario / 6–7 = superior

if "NIVEL_ED" in df.columns:
    df["edu_secundario"] = df["NIVEL_ED"].isin([4,5]).astype(int)
    df["edu_superior"]   = df["NIVEL_ED"].isin([6,7]).astype(int)
    df.drop(columns=["NIVEL_ED"], inplace=True)

# 5. CONDICIÓN DE ACTIVIDAD (ESTADO) → desocupado / inactivo
# 1=ocupado / 2=desocupado / 3=inactivo / 4=menor10

if "ESTADO" in df.columns:
    df["desocupado"] = (df["ESTADO"] == 2).astype(int)
    df["inactivo"]   = df["ESTADO"].isin([3,4]).astype(int)
    df.drop(columns=["ESTADO"], inplace=True)

# 6. CATEGORÍA DE INACTIVIDAD (CAT_INAC)
# 1–2=jubilado/rentista / 3=estudiante / 4=ama de casa / 5–7=otros

if "CAT_INAC" in df.columns:
    df["inac_jubilado"]  = df["CAT_INAC"].isin([1,2]).astype(int)
    df["inac_estudiante"]= (df["CAT_INAC"] == 3).astype(int)
    df["inac_amacasa"]   = (df["CAT_INAC"] == 4).astype(int)
    df["inac_otros"]     = df["CAT_INAC"].isin([5,6,7]).astype(int)
    df.drop(columns=["CAT_INAC"], inplace=True)

# 7. ALFABETISMO (CH09) → analfabeto
# 1=sabe leer / 2 o 3=no

if "CH09" in df.columns:
    df["analfabeto"] = df["CH09"].isin([2,3]).astype(int)
    df.drop(columns=["CH09"], inplace=True)

# 8. PARENTESCO (CH03)
# 1=jefe / 2–3=cónyuge o hijo / 4–10=otros

if "CH03" in df.columns:
    df["paren_conyuge_hijo"] = df["CH03"].isin([2,3]).astype(int)
    df["paren_otro"]         = df["CH03"].isin([4,5,6,7,8,9,10]).astype(int)
    df.drop(columns=["CH03"], inplace=True)

# 9. AGLOMERADO → GBA / CABA

if "AGLOMERADO" in df.columns:
    df["GBA"]  = (df["AGLOMERADO"] == 33).astype(int)
    df["CABA"] = (df["AGLOMERADO"] == 32).astype(int)
    df.drop(columns=["AGLOMERADO"], inplace=True)

# 10. Columnas numéricas que se mantienen:
# ANO4, CH06 (edad), PP3E_TOT (horas)

cols_final = ["ANO4", "CH06", "PP3E_TOT"] + [c for c in df.columns if c not in ["ANO4","CH06","PP3E_TOT"]]
prueba_final = df[cols_final].copy()

print("Forma original:", prueba.shape, "→ Forma final:", prueba_final.shape)
prueba_final.head()


**A. Enfoque de validación**  

In [ ]:
assert "ANO4" in prueba_final.columns, "Falta la columna ANO4 en 'prueba_final'."
assert "pobreza" in prueba_final.columns, "Falta la columna 'pobreza' en 'prueba_final'."

splits = {}  # guardamos aquí los splits por año

for year in sorted(prueba_final["ANO4"].dropna().unique()):
    df_y = prueba_final.loc[prueba["ANO4"] == year].copy()

    # Vector objetivo
    y = df_y["pobreza"].astype(int)

    # Matriz de features: resto de columnas numéricas (excluye 'pobreza')
    X = df_y.drop(columns=["pobreza"])

    # Nos quedamos solo con columnas numéricas para evitar errores en modelos
    X = X.select_dtypes(include=[np.number]).copy()

    # Agregar columna de unos para el intercepto (al final)
    X["intercepto"] = 1.0

    # 70/30 con semilla 444
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=444
    )

    splits[year] = {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test
    }

    # Resumen rápido
    print(f"Año {year}: X_train {X_train.shape}, X_test {X_test.shape}, "
          f"y_train={y_train.shape}, y_test={y_test.shape}, "
          f"p(1) train={y_train.mean():.3f}, p(1) test={y_test.mean():.3f}")

X_train_2005 = splits[2005]["X_train"]
y_train_2005 = splits[2005]["y_train"]
X_test_2005  = splits[2005]["X_test"]
y_test_2005  = splits[2005]["y_test"]

X_train_2025 = splits[2025]["X_train"]
y_train_2025 = splits[2025]["y_train"]
X_test_2025  = splits[2025]["X_test"]
y_test_2025  = splits[2025]["y_test"]


EJERCICIO 1

In [ ]:
year = 2005

# Extraer bases
X_train = splits[year]["X_train"].copy()
X_test  = splits[year]["X_test"].copy()

# --- Excluir la variable ANO4 de la comparación 
cols = [c for c in X_train.columns if c != "ANO4"]

# === Calcular medias y diferencias ===
tabla_diff = pd.DataFrame({
    "Media_train": X_train[cols].mean(),
    "Media_test":  X_test[cols].mean()
})

# Diferencia simple
tabla_diff["Dif_media"] = tabla_diff["Media_train"] - tabla_diff["Media_test"]

#Test de diferencia de medias 
p_values = []
for col in cols:
    try:
        stat, p = stats.ttest_ind(X_train[col], X_test[col], equal_var=False, nan_policy="omit")
        p_values.append(p)
    except Exception:
        p_values.append(np.nan)

tabla_diff["p_value"] = p_values
tabla_diff["significativa_5%"] = tabla_diff["p_value"] < 0.05

# Ordenar por magnitud de diferencia
tabla_diff = tabla_diff.sort_values("Dif_media", key=abs, ascending=False)

# Mostrar tabla completa 
pd.set_option("display.max_rows", None)
display(tabla_diff.round(4))


In [ ]:
year = 2025

# Extraer bases
X_train = splits[year]["X_train"].copy()
X_test  = splits[year]["X_test"].copy()

# --- Excluir la variable ANO4 de la comparación 
cols = [c for c in X_train.columns if c != "ANO4"]

# Calcular medias y diferencias 
tabla_diff = pd.DataFrame({
    "Media_train": X_train[cols].mean(),
    "Media_test":  X_test[cols].mean()
})

# Diferencia simple
tabla_diff["Dif_media"] = tabla_diff["Media_train"] - tabla_diff["Media_test"]

# Test de diferencia de medias 
p_values = []
for col in cols:
    try:
        stat, p = stats.ttest_ind(X_train[col], X_test[col], equal_var=False, nan_policy="omit")
        p_values.append(p)
    except Exception:
        p_values.append(np.nan)

tabla_diff["p_value"] = p_values
tabla_diff["significativa_5%"] = tabla_diff["p_value"] < 0.05

# Ordenar por magnitud de diferencia
tabla_diff = tabla_diff.sort_values("Dif_media", key=abs, ascending=False)

# Mostrar tabla completa 
pd.set_option("display.max_rows", None)
display(tabla_diff.round(4))


EJERCICIO 2

In [ ]:
respondieron_2005 = prueba_final[prueba_final["ANO4"] == 2005].copy()
respondieron_2025 = prueba_final[prueba_final["ANO4"] == 2025].copy()

In [ ]:
#cargamos la base de datos NO respondieron
norespondieron = pd.read_excel("norespondieron.xlsx")

In [ ]:
norespondieron_2005 = norespondieron[norespondieron["ANO4"] == 2005].copy()
norespondieron_2025 = norespondieron[norespondieron["ANO4"] == 2025].copy()

**TRABAJO PRÁCTICO 4**

**A. Modelo de Regresion Logistica con Regularización: Ridge y LASSO**


1. Visualización

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

# 0) Preprocesamiento de tus bases 2025

# Solo variables numéricas
X_num = X_train_2025.select_dtypes(include=[np.number])

# Imputación de NaN con mediana
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_num)

# Guardamos nombres de columnas
feature_names = X_num.columns

# Vector de respuesta
y_arr = np.array(y_train_2025)

# 1) Grilla de lambdas: λ = 10^n para n = -5,...,5

n_values = np.arange(-5, 6)
lambdas = 10.0 ** n_values      
C_values = 1 / lambdas          

coef_lasso = []
coef_ridge = []

# 2) Ajustar LASSO y RIDGE para cada valor de C

for C in C_values:
    # -------- LASSO (L1) --------
    model_l1 = LogisticRegression(
        penalty='l1',
        C=C,
        solver='liblinear',
        max_iter=3000
    )
    model_l1.fit(X_train_imp, y_arr)
    coef_lasso.append(model_l1.coef_.flatten())
    
    # -------- RIDGE (L2) --------
    model_l2 = LogisticRegression(
        penalty='l2',
        C=C,
        solver='lbfgs',
        max_iter=3000
    )
    model_l2.fit(X_train_imp, y_arr)
    coef_ridge.append(model_l2.coef_.flatten())

coef_lasso = np.array(coef_lasso)
coef_ridge = np.array(coef_ridge)

# 3) Graficar trayectorias de coeficientes

plt.figure(figsize=(14, 6))

# --- Panel 1: LASSO ---
plt.subplot(1, 2, 1)
for i in range(coef_lasso.shape[1]):
    plt.plot(n_values, coef_lasso[:, i], alpha=0.7)
plt.title("Trayectorias de coeficientes - Penalidad LASSO (L1)")
plt.xlabel("n en λ = 10^n  (→ derecha = mayor penalidad)")
plt.ylabel("Valor del coeficiente")
plt.grid(True)

# --- Panel 2: RIDGE ---
plt.subplot(1, 2, 2)
for i in range(coef_ridge.shape[1]):
    plt.plot(n_values, coef_ridge[:, i], alpha=0.7)
plt.title("Trayectorias de coeficientes - Penalidad Ridge (L2)")
plt.xlabel("n en λ = 10^n  (→ derecha = mayor penalidad)")
plt.grid(True)

plt.tight_layout()
plt.show()


2. Penalidad óptima por Cross-validation y visualización

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold

# 0) Preprocesamiento (numéricas + imputación)
X_num = X_train_2025.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_num)
y_arr = np.array(y_train_2025)

# 1) Grilla de λ: λ = 10^n, n = -5,...,5
#    y su C asociado (C = 1/λ)

n_values = np.arange(-5, 6)       
lambdas = 10.0 ** n_values
C_values = 1.0 / lambdas

# 2) Cross-validation manual (5-fold estratificado)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=444)

err_l1_list = []   
err_l2_list = []

for C in C_values:
    fold_errors_l1 = []
    fold_errors_l2 = []
    
    for train_idx, val_idx in skf.split(X_train_imp, y_arr):
        X_tr, X_val = X_train_imp[train_idx], X_train_imp[val_idx]
        y_tr, y_val = y_arr[train_idx], y_arr[val_idx]
        
        # ----- LASSO (L1) -----
        model_l1 = LogisticRegression(
            penalty='l1',
            C=C,
            solver='liblinear',
            max_iter=3000
        )
        model_l1.fit(X_tr, y_tr)
        pred_l1 = model_l1.predict(X_val)
        err_l1 = np.mean(pred_l1 != y_val)   # error de clasificación
        fold_errors_l1.append(err_l1)
        
        # ----- RIDGE (L2) -----
        model_l2 = LogisticRegression(
            penalty='l2',
            C=C,
            solver='lbfgs',
            max_iter=3000
        )
        model_l2.fit(X_tr, y_tr)
        pred_l2 = model_l2.predict(X_val)
        err_l2 = np.mean(pred_l2 != y_val)
        fold_errors_l2.append(err_l2)
    
    err_l1_list.append(fold_errors_l1)
    err_l2_list.append(fold_errors_l2)

err_l1_arr = np.array(err_l1_list)  
err_l2_arr = np.array(err_l2_list)

# 3) λ óptimo (mínimo error promedio)

mean_err_l1 = err_l1_arr.mean(axis=1)
mean_err_l2 = err_l2_arr.mean(axis=1)

idx_best_l1 = np.argmin(mean_err_l1)
idx_best_l2 = np.argmin(mean_err_l2)

best_lambda_l1 = lambdas[idx_best_l1]
best_lambda_l2 = lambdas[idx_best_l2]

print("LASSO (L1):")
print(f"  λ óptimo: {best_lambda_l1:.6f} (n = {n_values[idx_best_l1]})")
print(f"  Error medio CV: {mean_err_l1[idx_best_l1]:.4f}")

print("\nRIDGE (L2):")
print(f"  λ óptimo: {best_lambda_l2:.6f} (n = {n_values[idx_best_l2]})")
print(f"  Error medio CV: {mean_err_l2[idx_best_l2]:.4f}")

# 4) Boxplots de error de clasificación por λ

lambda_labels = [f"10^{n}" for n in n_values]

plt.figure(figsize=(16, 6))

# ----- LASSO -----
plt.subplot(1, 2, 1)
plt.boxplot(err_l1_list, positions=range(len(C_values)))
plt.xticks(ticks=range(len(C_values)), labels=lambda_labels, rotation=45)
plt.xlabel("λ (escala logarítmica: 10^n)")
plt.ylabel("Error de clasificación (1 - accuracy)")
plt.title("LASSO (L1) - Error de validación por λ")
plt.grid(True, axis='y')

# ----- RIDGE -----
plt.subplot(1, 2, 2)
plt.boxplot(err_l2_list, positions=range(len(C_values)))
plt.xticks(ticks=range(len(C_values)), labels=lambda_labels, rotation=45)
plt.xlabel("λ (escala logarítmica: 10^n)")
plt.ylabel("Error de clasificación (1 - accuracy)")
plt.title("Ridge (L2) - Error de validación por λ")
plt.grid(True, axis='y')

plt.tight_layout()
plt.show()


OPCIONAL

In [ ]:
zero_props = []

for C in C_values:
    model_l1 = LogisticRegression(
        penalty='l1',
        C=C,
        solver='liblinear',
        max_iter=3000
    )
    model_l1.fit(X_train_imp, y_arr)
    coefs = model_l1.coef_.flatten()
    zero_props.append(np.mean(coefs == 0))

plt.figure(figsize=(7, 5))
plt.plot(n_values, zero_props, marker='o')
plt.xlabel("n en λ = 10^n (→ derecha = mayor penalidad)")
plt.ylabel("Proporción de coeficientes = 0")
plt.title("LASSO: proporción de variables 'apagadas' según λ")
plt.grid(True)
plt.show()


3. Estimación con λ y comparación de coeficientes

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

# 0) Preprocesamiento (numéricas + imputación)

X_num = X_train_2025.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_num)
y_arr = np.array(y_train_2025)

feature_names = X_num.columns

# 1) Recuperar lambdas óptimos del paso anterior

C_l1 = 1.0 / best_lambda_l1
C_l2 = 1.0 / best_lambda_l2

print(f"C L1 óptimo: {C_l1}")
print(f"C L2 óptimo: {C_l2}")

# ======================================================
# 2) Estimar los 3 modelos
# ======================================================

# --- Modelo sin penalidad ---
model_none = LogisticRegression(
    penalty=None,       # SIN penalización
    solver='lbfgs',
    max_iter=3000
)

model_none.fit(X_train_imp, y_arr)
coef_none = model_none.coef_.flatten()

# --- Modelo LASSO (L1) ---
model_l1 = LogisticRegression(
    penalty='l1',
    C=C_l1,
    solver='liblinear',
    max_iter=3000
)
model_l1.fit(X_train_imp, y_arr)
coef_l1 = model_l1.coef_.flatten()

# --- Modelo Ridge (L2) ---
model_l2 = LogisticRegression(
    penalty='l2',
    C=C_l2,
    solver='lbfgs',
    max_iter=3000
)
model_l2.fit(X_train_imp, y_arr)
coef_l2 = model_l2.coef_.flatten()

# 3) Construir tabla comparativa

coef_table = pd.DataFrame({
    'variable': feature_names,
    'coef_sin_penalidad': coef_none,
    'coef_L1': coef_l1,
    'coef_L2': coef_l2
})

coef_table


**B. Árboles**

4. Estimen un árbol de decisión podado (CART)]

In [ ]:
#OPCIÓN 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold

# Preprocesamiento (igual que antes)
X_num = X_train_2025.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_num)
y_arr = np.array(y_train_2025)

# 1) Obtener la grilla de ccp_alphas desde el árbol completo

tree_full = DecisionTreeClassifier(random_state=444)
path = tree_full.cost_complexity_pruning_path(X_train_imp, y_arr)

ccp_alphas = path.ccp_alphas

# Muchas veces el último alpha deja el árbol en una sola hoja → lo excluimos
ccp_alphas = ccp_alphas[:-1]

print(f"Número de valores de ccp_alpha a evaluar: {len(ccp_alphas)}")

# 2) 10-fold Cross-Validation para cada ccp_alpha

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=444)

mean_errors = []

for alpha in ccp_alphas:
    fold_errors = []
    
    for train_idx, val_idx in skf.split(X_train_imp, y_arr):
        X_tr, X_val = X_train_imp[train_idx], X_train_imp[val_idx]
        y_tr, y_val = y_arr[train_idx], y_arr[val_idx]
        
        tree = DecisionTreeClassifier(
            random_state=444,
            ccp_alpha=alpha
        )
        tree.fit(X_tr, y_tr)
        y_pred = tree.predict(X_val)
        
        err = np.mean(y_pred != y_val)   
        fold_errors.append(err)
    
    mean_errors.append(np.mean(fold_errors))

mean_errors = np.array(mean_errors)

# α óptimo (el que minimiza el error medio)
best_idx = np.argmin(mean_errors)
best_alpha = ccp_alphas[best_idx]
best_err = mean_errors[best_idx]

print(f"\nccp_alpha óptimo: {best_alpha}")
print(f"Error medio de clasificación (10-fold CV) en ese alpha: {best_err:.4f}")

# 3) Gráfico: error de clasificación vs ccp_alpha  

plt.figure(figsize=(8, 5))
plt.plot(ccp_alphas, mean_errors, marker='o')
plt.axvline(best_alpha, linestyle='--', label=f"ccp_alpha óptimo = {best_alpha:.4g}")
plt.xlabel("ccp_alpha (costo de complejidad)")
plt.ylabel("Error de clasificación medio (10-fold CV)")
plt.title("CART podado: error de validación vs ccp_alpha")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree

# 1) Entrenar el árbol podado definitivo

tree_pruned = DecisionTreeClassifier(
    random_state=444,
    ccp_alpha=best_alpha
)
tree_pruned.fit(X_train_imp, y_arr)

# 2) Panel A: Gráfico del árbol podado

plt.figure(figsize=(20, 10))
plot_tree(
    tree_pruned,
    feature_names=X_num.columns,
    class_names=["no pobre", "pobre"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Panel A: Árbol de Decisión Podado (CART)")
plt.show()

# 3) Panel B: Importancia de las variables

importances = tree_pruned.feature_importances_
sorted_idx = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.barh(
    X_num.columns[sorted_idx],
    importances[sorted_idx],
    color="teal"
)
plt.xlabel("Importancia")
plt.title("Panel B: Importancia de Predictores en el Árbol Podado")
plt.show()


In [ ]:
#SEGUNDA OPCIÓN

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer

# Preprocesamiento 
X_num = X_train_2025.select_dtypes(include=[np.number])
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_num)
y_arr = np.array(y_train_2025)

# Árbol completo para obtener la grilla de ccp_alpha
tree_full = DecisionTreeClassifier(random_state=444)
path = tree_full.cost_complexity_pruning_path(X_train_imp, y_arr)
ccp_alphas = path.ccp_alphas[:-1] 

print("Voy a evaluar", len(ccp_alphas), "valores de ccp_alpha")

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=444)

mean_errors = []
n_leaves_list = []

for alpha in ccp_alphas:
    fold_errors = []
    
    for train_idx, val_idx in skf.split(X_train_imp, y_arr):
        X_tr, X_val = X_train_imp[train_idx], X_train_imp[val_idx]
        y_tr, y_val = y_arr[train_idx], y_arr[val_idx]
        
        tree = DecisionTreeClassifier(random_state=444, ccp_alpha=alpha)
        tree.fit(X_tr, y_tr)
        y_pred = tree.predict(X_val)
        err = np.mean(y_pred != y_val)
        fold_errors.append(err)
    
    mean_errors.append(np.mean(fold_errors))
    
    # número de hojas con TODO el train (solo para ver complejidad)
    tree_tmp = DecisionTreeClassifier(random_state=444, ccp_alpha=alpha)
    tree_tmp.fit(X_train_imp, y_arr)
    n_leaves_list.append(tree_tmp.get_n_leaves())

mean_errors = np.array(mean_errors)
n_leaves_list = np.array(n_leaves_list)

# alpha con mínimo error (sin restricción)
idx_best = np.argmin(mean_errors)
alpha_best_raw = ccp_alphas[idx_best]

# ahora elegimos el alpha con menor error PERO con al menos 5 hojas
candidatos = np.where(n_leaves_list >= 5)[0]
if len(candidatos) > 0:
    idx_best_con_hojas = candidatos[np.argmin(mean_errors[candidatos])]
    best_alpha = ccp_alphas[idx_best_con_hojas]
else:
    # si ningún árbol tiene >=5 hojas, usamos el mejor sin restricción
    best_alpha = alpha_best_raw

print("alpha con mínimo error (sin restricción):", alpha_best_raw)
print("alpha elegido (error bajo y >=5 hojas):", best_alpha)

print("Hojas para cada alpha (primeros 10):")
for a, h, e in list(zip(ccp_alphas, n_leaves_list, mean_errors))[:10]:
    print(f"alpha={a:.5g}, hojas={h}, error={e:.4f}")

#esto no es el grafico que me deberia estar dando. 
plt.figure(figsize=(8,5))
plt.plot(ccp_alphas, mean_errors, marker='o')
plt.xlabel("ccp_alpha (costo de complejidad)")
plt.ylabel("Error de clasificación medio (10-fold CV)")
plt.title("Error de clasificación vs ccp_alpha (CART podado)")
plt.grid(True)
plt.show()


5. Visualización del árbol podado por cross-validation

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

tree_pruned = DecisionTreeClassifier(
    random_state=444,
    ccp_alpha=best_alpha
)
tree_pruned.fit(X_train_imp, y_arr)

print("Nodos:", tree_pruned.tree_.node_count)
print("Hojas:", tree_pruned.get_n_leaves())
print("Profundidad:", tree_pruned.get_depth())

# Panel A: árbol
plt.figure(figsize=(20, 10))
plot_tree(
    tree_pruned,
    feature_names=X_num.columns,
    class_names=["no pobre", "pobre"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Panel A: Árbol de Decisión Podado (CART)")
plt.show()

# Panel B: importancia de variables
importances = tree_pruned.feature_importances_
sorted_idx = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.barh(X_num.columns[sorted_idx], importances[sorted_idx])
plt.xlabel("Importancia")
plt.title("Panel B: Importancia de Predictores en el Árbol Podado")
plt.show()


**C. Comparación entre métodos**

Ejercicio 6

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

# Solo variables numéricas (igual que antes)
X_num_train = X_train_2025.select_dtypes(include=[np.number])
X_num_test  = X_test_2025.select_dtypes(include=[np.number])

# Imputación
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_num_train)
X_test_imp  = imputer.transform(X_num_test)

y_train_arr = np.array(y_train_2025)
y_test_arr  = np.array(y_test_2025)


In [ ]:
from sklearn.linear_model import LogisticRegression

logit_none = LogisticRegression(penalty=None, solver='lbfgs', max_iter=3000)
logit_none.fit(X_train_imp, y_train_arr)
pred_proba_none = logit_none.predict_proba(X_test_imp)[:,1]
pred_none = (pred_proba_none > 0.5).astype(int)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

K_opt = 30  
knn = KNeighborsClassifier(n_neighbors=K_opt)
knn.fit(X_train_imp, y_train_arr)
pred_knn = knn.predict(X_test_imp)
pred_proba_knn = knn.predict_proba(X_test_imp)[:,1]


In [ ]:
from sklearn.linear_model import LogisticRegression

C_l1 = 1 / best_lambda_l1   

logit_l1 = LogisticRegression(penalty='l1', C=C_l1, solver='liblinear', max_iter=3000)
logit_l1.fit(X_train_imp, y_train_arr)
pred_proba_l1 = logit_l1.predict_proba(X_test_imp)[:,1]
pred_l1 = (pred_proba_l1 > 0.5).astype(int)


In [ ]:
C_l2 = 1 / best_lambda_l2

logit_l2 = LogisticRegression(penalty='l2', C=C_l2, solver='lbfgs', max_iter=3000)
logit_l2.fit(X_train_imp, y_train_arr)
pred_proba_l2 = logit_l2.predict_proba(X_test_imp)[:,1]
pred_l2 = (pred_proba_l2 > 0.5).astype(int)


In [ ]:
tree_pruned = DecisionTreeClassifier(random_state=444, ccp_alpha=best_alpha)
tree_pruned.fit(X_train_imp, y_train_arr)
pred_tree = tree_pruned.predict(X_test_imp)
pred_proba_tree = tree_pruned.predict_proba(X_test_imp)[:,1]


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, accuracy_score

def evaluar_modelo(y_true, pred, pred_proba):
    cm = confusion_matrix(y_true, pred)
    acc = accuracy_score(y_true, pred)
    err = 1 - acc
    auc = roc_auc_score(y_true, pred_proba)

    return cm, acc, err, auc


In [ ]:
resultados = {}

modelos = {
    "Logit_sin_penalidad": (pred_none, pred_proba_none),
    "KNN": (pred_knn, pred_proba_knn),
    "LASSO": (pred_l1, pred_proba_l1),
    "Ridge": (pred_l2, pred_proba_l2),
    "Árbol_podado": (pred_tree, pred_proba_tree)
}

for nombre, (pred, proba) in modelos.items():
    cm, acc, err, auc = evaluar_modelo(y_test_arr, pred, proba)
    resultados[nombre] = {
        "confusion_matrix": cm,
        "accuracy": acc,
        "1-error": err,
        "AUC": auc
    }

resultados


In [ ]:
import pandas as pd

tabla = pd.DataFrame({
    modelo: {
        "Accuracy": r["accuracy"],
        "1-Error": r["1-error"],
        "AUC": r["AUC"]
    }
    for modelo, r in resultados.items()
}).T

tabla

In [ ]:
modelos_conf = {
    "Logit sin penalidad": resultados["Logit_sin_penalidad"]["confusion_matrix"],
    "KNN (K óptimo)": resultados["KNN"]["confusion_matrix"],
    "LASSO (L1)": resultados["LASSO"]["confusion_matrix"],
    "Ridge (L2)": resultados["Ridge"]["confusion_matrix"],
    "Árbol podado": resultados["Árbol_podado"]["confusion_matrix"]
}

for nombre, matriz in modelos_conf.items():
    print("\n==============================")
    print(f"   {nombre}")
    print("==============================")
    print(matriz)

In [ ]:
plt.figure(figsize=(8,5))

for nombre, (pred, proba) in modelos.items():
    fpr, tpr, _ = roc_curve(y_test_arr, proba)
    plt.plot(fpr, tpr, label=nombre)

plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curvas ROC – Comparación de modelos")
plt.legend()
plt.grid(True)
plt.show()


Ejercicio 7

en el informe